# AIFM PE Buyout Fund — Risk Notebook

Risk analysis for the AIFM PE Buyout Fund under AIFMD.
Strategy: European mid-market buyout, 8 portfolio companies across technology,
healthcare, industrials, consumer and energy transition sectors.

Regulatory framework:
- AIFMD: Directive 2011/61/EU
- AIFMD II: Directive 2024/927/EU
- Delegated Regulation: EU 231/2013
- Annex IV reporting: EU 231/2013, ESMA technical guidance v1.7 (July 2024)
- Annex VI stress testing: ESMA/2020/1498
- Luxembourg implementation: Law of 12 July 2013 on AIFMs (AIFM Law)
- IPEV Valuation Guidelines (International Private Equity Valuation)
- ILPA reporting standards (investor reporting best practice)

Dual UCITS/AIFM ManCo:
- CSSF Regulation 10-04 (organisational and prudential requirements)
- CSSF Regulation 22-05 (sustainability requirements, amending 10-04)

#### In this notebook

AIFM PE Buyout Fund I. Vintage 2018, EUR 200m target size, 10-year fund life.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.plot_style import ACCENT, ACCENT2, ACCENT3
from src.database import (
    get_engine, PEFund, PEPortfolioCompany,
    PEFundInvestment, PECashFlow, PENavHistory
)
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.esg_utils import build_esg_df, esg_portfolio_summary, ESG_FIELDS, ESG_THRESHOLD
from sqlalchemy.orm import Session

FUND_ID = 'AIFM_PE_Buyout'
TODAY   = '2026-05-13'
ENGINE  = get_engine()
BBG     = Bloomberg()

## 1. Load and Validate Fund Data

PE fund data is structured differently from liquid funds. There is no daily position
snapshot. Data is organised around portfolio companies, cash flows, and quarterly NAV
appraisals, stored in dedicated PE tables in SQLite.

The data flow is:

`generate_pe_fund.py` → SQLite PE tables → notebook

Tables loaded:
- `pe_funds`: fund metadata (vintage, size, investment period)
- `pe_portfolio_companies`: company master data (sector, country, stage)
- `pe_fund_investments`: fund-specific investment data (cost basis, ownership, exit)
- `pe_cash_flows`: capital calls, distributions, fees, exit proceeds
- `pe_nav_history`: quarterly NAV per portfolio company and at fund level

In production, these tables would be populated from the fund administrator system
(Investran, eFront, Allvue) rather than a synthetic generator.

In [ ]:
# MRS-65: Load and validate PE fund data
with Session(ENGINE) as session:
    fund        = session.query(PEFund).filter_by(fund_id=FUND_ID).first()
    companies   = session.query(PEPortfolioCompany).all()
    investments = session.query(PEFundInvestment).filter_by(fund_id=FUND_ID).all()
    cash_flows  = session.query(PECashFlow).filter_by(fund_id=FUND_ID).all()
    nav_history = session.query(PENavHistory).filter_by(fund_id=FUND_ID).all()

cf_df  = pd.DataFrame([{
    'date'       : cf.date,
    'company_id' : cf.company_id,
    'flow_type'  : cf.flow_type,
    'amount_eur' : cf.amount_eur,
    'description': cf.description,
} for cf in cash_flows])
cf_df['date'] = pd.to_datetime(cf_df['date'])

nav_df = pd.DataFrame([{
    'date'           : n.date,
    'company_id'     : n.company_id,
    'nav_eur'        : n.nav_eur,
    'gross_multiple' : n.gross_multiple,
    'unrealised_gain': n.unrealised_gain,
    'cost_basis_eur' : n.cost_basis_eur,
} for n in nav_history])
nav_df['date'] = pd.to_datetime(nav_df['date'])

inv_df = pd.DataFrame([{
    'company_id'     : i.company_id,
    'investment_date': i.investment_date,
    'cost_basis_eur' : i.cost_basis_eur,
    'ownership_pct'  : i.ownership_pct,
    'entry_multiple' : i.entry_multiple,
    'exit_date'      : i.exit_date,
    'exit_price_eur' : i.exit_price_eur,
    'exit_multiple'  : i.exit_multiple,
} for i in investments])

co_df = pd.DataFrame([{
    'company_id'      : c.company_id,
    'company_name'    : c.company_name,
    'sector'          : c.sector,
    'country'         : c.country,
    'investment_stage': c.investment_stage,
    'status'          : c.status,
} for c in companies])

total_called      = cf_df[cf_df['amount_eur'] < 0]['amount_eur'].sum()
total_distributed = cf_df[cf_df['amount_eur'] > 0]['amount_eur'].sum()
current_nav       = nav_df[nav_df['company_id'].isna()]['nav_eur'].sort_values().iloc[-1]
active            = [i for i in investments if i.exit_date is None]
exited            = [i for i in investments if i.exit_date is not None]
NAV               = current_nav

# 24 cash flows loaded - internal data quality check
print(f"Fund              : {fund.fund_name}")
print(f"Vintage           : {fund.vintage_year}")
print(f"Strategy          : {fund.strategy}")
print(f"Target size       : EUR {fund.target_size_eur:,.0f}")
print(f"Fund life         : {fund.fund_life_years} years")
print(f"Investment period : ends {fund.investment_period_end}")
print()
print(f"Portfolio companies : {len(companies)}")
print(f"  Active            : {len(active)}")
print(f"  Exited            : {len(exited)}")
print()
print(f"Total called      : EUR {abs(total_called):,.0f}")
print(f"Total distributed : EUR {total_distributed:,.0f}")
print(f"Current NAV       : EUR {current_nav:,.0f}")
print(f"Quarterly valuations : {len([n for n in nav_history if n.company_id is not None])}")

In [ ]:
# portfolio company table
portfolio = co_df.merge(inv_df, on='company_id')
portfolio['investment_date'] = pd.to_datetime(portfolio['investment_date'])
portfolio['exit_date']       = pd.to_datetime(portfolio['exit_date'])

portfolio_display = portfolio[[
    'company_name', 'sector', 'country', 'investment_stage',
    'status', 'investment_date', 'cost_basis_eur',
    'ownership_pct', 'entry_multiple', 'exit_date', 'exit_multiple'
]].copy()

portfolio_display['investment_date'] = portfolio_display['investment_date'].dt.strftime('%Y-%m-%d')
portfolio_display['exit_date']       = portfolio_display['exit_date'].dt.strftime('%Y-%m-%d').fillna('—')
portfolio_display['exit_multiple']   = portfolio_display['exit_multiple'].map(
    lambda x: f'{x:.2f}x' if pd.notna(x) else '—')
portfolio_display['cost_basis_eur']  = portfolio_display['cost_basis_eur'].map('{:,.0f}'.format)
portfolio_display['ownership_pct']   = portfolio_display['ownership_pct'].map('{:.1f}%'.format)

portfolio_display.columns = [
    'Company', 'Sector', 'Country', 'Stage',
    'Status', 'Invested', 'Cost (EUR)',
    'Ownership', 'Entry Multiple', 'Exit Date', 'Exit Multiple'
]
portfolio_display.set_index('Company', inplace=True)
portfolio_display

## 2. Independent Appraisal and Valuation Reports

Unlike liquid funds where prices are observable daily in the market, PE portfolio
companies are valued quarterly by an independent appraisal firm. The ManCo cannot
self-certify valuations under AIFMD Article 19 without independent oversight.

**Who produces this data**: independent valuation firms (KPMG, Duff & Phelps,
Lincoln International) appointed by the fund's board. Data arrives quarterly as a
structured valuation report, stored here in `pe_valuation_report`.

**What the report contains:**
- Appraised NAV: equity value after deducting net debt from enterprise value
- LTM financials: revenue, EBITDA, margins from management accounts
- Enterprise value and EV/EBITDA multiple used by the appraiser
- Net debt and leverage ratio
- Discount rate (WACC) used in income approach valuations
- Covenant status and headroom
- Key risks identified by the appraiser

**Covenant types vary by company stage:**

*Buyout companies (PE_001, PE_002, PE_003, PE_004, PE_007, PE_008)*: maintenance
covenants tested quarterly against LTM EBITDA:

$$\text{Leverage ratio} = \frac{\text{Net Debt}}{EBITDA_{LTM}} \leq \text{Leverage covenant}$$

$$\text{Coverage ratio} = \frac{EBITDA_{LTM}}{\text{Interest expense}} \geq \text{Coverage covenant}$$

$$\text{Leverage headroom} = \frac{\text{Leverage covenant} - \text{Leverage ratio}}{\text{Leverage covenant}}$$

*Growth companies (PE_005 EnergyTrans, PE_006 FinTech Nordic)*: revenue and
liquidity covenants since EBITDA may be negative in early years:

$$\text{Revenue covenant}: \text{LTM Revenue} \geq \text{Minimum threshold}$$
$$\text{Cash covenant}: \text{Cash balance} \geq \text{Minimum cash}$$

For SaaS and subscription models, ARR (Annual Recurring Revenue) is tracked
as the primary growth metric alongside cash covenant.

> **Note**: unlike listed REITs where market prices update daily, direct PE
> valuations are quarterly and reflect appraiser judgment, not market transactions.
> Valuation risk (model risk) is therefore a significant component of PE risk,
> distinct from market risk in liquid portfolios.

In [ ]:
from src.database import PEValuationReport

with Session(ENGINE) as session:
    val_reports = session.query(PEValuationReport).filter_by(fund_id=FUND_ID).all()

vr_df = pd.DataFrame([{
    'company_id'        : v.company_id,
    'date'              : v.date,
    'appraised_nav_eur' : v.appraised_nav_eur,
    'ebitda_ltm_eur'    : v.ebitda_ltm_eur,
    'revenue_ltm_eur'   : v.revenue_ltm_eur,
    'ebitda_margin'     : v.ebitda_margin,
    'net_debt_eur'      : v.net_debt_eur,
    'ev_eur'            : v.ev_eur,
    'ev_ebitda'         : v.ev_ebitda,
    'leverage_ratio'    : v.leverage_ratio,
    'leverage_covenant' : v.leverage_covenant,
    'coverage_ratio'    : v.coverage_ratio,
    'coverage_covenant' : v.coverage_covenant,
    'covenant_type'     : v.covenant_type,
    'appraiser'         : v.appraiser,
    'key_risks'         : v.key_risks,
    'arr_eur'           : v.arr_eur,
} for v in val_reports])
vr_df['date'] = pd.to_datetime(vr_df['date'])

print(f"Valuation reports loaded : {len(vr_df)}")
print(f"Companies covered        : {vr_df['company_id'].nunique()}")
print(f"Date range               : {vr_df['date'].min().date()} to {vr_df['date'].max().date()}")
print()

def format_leverage(row):
    if pd.isna(row['leverage_ratio']) or row['leverage_ratio'] > 50:
        return 'N/A (neg. EBITDA)'
    return f"{row['leverage_ratio']:.2f}x"

def covenant_headroom(row):
    if pd.isna(row['leverage_covenant']) or pd.isna(row['leverage_ratio']):
        return '—'
    if row['leverage_ratio'] > 50:
        return 'BREACH'
    headroom = (row['leverage_covenant'] - row['leverage_ratio']) / row['leverage_covenant'] * 100
    if headroom < 0:
        return f'BREACH ({headroom:.1f}%)'
    elif headroom < 20:
        return f'⚠ {headroom:.1f}%'
    return f'{headroom:.1f}%'

latest = vr_df.sort_values('date').groupby('company_id').last().reset_index()
latest['leverage_display'] = latest.apply(format_leverage, axis=1)
latest['headroom']         = latest.apply(covenant_headroom, axis=1)

print(latest[['company_id', 'appraised_nav_eur', 'leverage_display', 
              'leverage_covenant', 'headroom', 'covenant_type']].to_string())